In [ ]:
import json
import os
import glob
from rdflib import Graph
import owlrl

ALL_CONTRACT_IDS = {f"C{i}" for i in range(1, 11)}
SCENARIO_FOLDER = "../dataset/scenarios_eval"
ONTOLOGY_PATH = "../dataset/ontologies_eval"
TEST_SCENS_PATH = os.path.join(SCENARIO_FOLDER, "test_scens.json")


def format_term(term):
    """Cleans RDF terms for readable console output."""
    if term is None:
        return "N/A"
    val = str(term.toPython())
    return val.split('#')[-1] if 'http' in val else val


def print_pretty_sparql(query_text):
    """Formats the SPARQL for debugging failing sections."""
    fmt = query_text.replace('\\n', '\n').replace('\n', ' ')
    keywords = ["SELECT", "WHERE", "FILTER", "OPTIONAL", "UNION"]
    for kw in keywords:
        fmt = fmt.replace(kw, f"\n{kw}")
    print("\n   [ DEBUG: SPARQL QUERY ]")
    print("   " + fmt.strip().replace('\n', '\n   '))
    print("   " + "-" * 30)


def print_section(label, expected_ids, found_ids, sparql_rows=None):
    """Prints status and mismatch details for a specific category."""
    is_match = set(expected_ids) == set(found_ids)
    icon = "✅" if is_match else "❌"

    print(f"\n   {icon} {label}:")

    if sparql_rows:
        for row in sparql_rows:
            print(f"      | {' | '.join(format_term(t) for t in row)}")
    else:
        print(f"      IDs: {sorted(list(found_ids)) if found_ids else 'None'}")

    if not is_match:
        missing = set(expected_ids) - set(found_ids)
        extra = set(found_ids) - set(expected_ids)
        if missing:
            print(f"      ⚠️  MISSING: {sorted(list(missing))}")
        if extra:
            print(f"      ⚠️  EXTRA:   {sorted(list(extra))}")


def load_test_queries(path):
    """Loads the overriding queries from test_scens.json, keyed by scenario_id."""
    if not os.path.exists(path):
        print(f"WARNING: {path} not found — using queries from scenario JSON files instead.\n")
        return {}
    with open(path, 'r') as f:
        data = json.load(f)
    return {entry['scenario_id']: entry for entry in data}


def load_graph():
    """Loads the TBox (if present) and all ABox files from the ontology folder."""
    print("=" * 80)
    print("INITIALIZING: Loading Ontologies & Reasoning...")
    g = Graph()

    # Load TBox if it exists
    tbox_path = os.path.join(ONTOLOGY_PATH, "life_insurance_tbox.ttl")
    if os.path.exists(tbox_path):
        g.parse(tbox_path, format="turtle")
        print(f"  Loaded TBox: {tbox_path}")
    else:
        print(f"  (No TBox found at {tbox_path}, skipping)")

    # Load all ABox files (any .ttl file that isn't the TBox)
    abox_files = sorted(glob.glob(os.path.join(ONTOLOGY_PATH, "*.ttl")))
    abox_files = [f for f in abox_files if os.path.basename(f) != "life_insurance_tbox.ttl"]

    if not abox_files:
        print("  WARNING: No ABox files found.")
    for abox_path in abox_files:
        g.parse(abox_path, format="turtle")
        print(f"  Loaded ABox: {abox_path}")

    print("  Running OWL-RL reasoning...")
    owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(g)
    print("READY.\n")
    return g


def compute_scores(scenarios, g, test_queries, scen_range):
    total = 0
    correct = 0
    results = []

    for scen in scenarios:
        sid = scen['scenario_id']
        try:
            num = int(sid.split('-')[-1])
        except:
            continue
        if not (scen_range[0] <= num <= scen_range[1]):
            continue

        if sid in test_queries:
            query_covered = test_queries[sid]['query_covered']
            query_denied  = test_queries[sid]['query_denied']
        else:
            query_covered = scen['query_covered']
            query_denied  = scen['query_denied']

        json_cov = {r['contract_id'] for r in scen['contract_responses'] if r['status'] == "COVERED"}
        json_den = {r['contract_id'] for r in scen['contract_responses'] if r['status'] == "DENIED"}
        json_na  = {r['contract_id'] for r in scen['contract_responses'] if r['status'] == "NOT_APPLICABLE"}

        found_cov = {str(row[0].toPython()) for row in g.query(query_covered) if row[0] is not None}
        found_den = {str(row[0].toPython()) for row in g.query(query_denied)  if row[0] is not None}
        found_na  = ALL_CONTRACT_IDS - (found_cov | found_den)

        # Build per-contract predicted vs ground-truth label
        gt_map   = {cid: "COVERED"        for cid in json_cov}
        gt_map  |= {cid: "DENIED"         for cid in json_den}
        gt_map  |= {cid: "NOT_APPLICABLE" for cid in json_na}

        pred_map  = {cid: "COVERED"        for cid in found_cov}
        pred_map |= {cid: "DENIED"         for cid in found_den}
        pred_map |= {cid: "NOT_APPLICABLE" for cid in found_na}

        scen_correct = sum(1 for cid in ALL_CONTRACT_IDS if pred_map.get(cid) == gt_map.get(cid))
        scen_total   = len(ALL_CONTRACT_IDS)  # always 10

        total   += scen_total
        correct += scen_correct
        results.append((sid, scen_correct, scen_total))

    pct = 100 * correct / total if total else 0
    return correct, total, pct, results


def print_scores(correct, total, pct, results):
    print("\n" + "="*80)
    print("SCORING SUMMARY")
    print("="*80)
    for sid, c, t in results:
        bar = "█" * c + "░" * (t - c)
        print(f"  {sid}:  {c:2d}/{t}  [{bar}]")
    print("-"*80)
    print(f"  TOTAL:  {correct}/{total}  →  {pct:.2f}%")
    print("="*80)



def load_scenarios():
    """Loads all scenario JSON files from the scenario folder, excluding test_scens.json."""
    files = glob.glob(os.path.join(SCENARIO_FOLDER, "*.json"))
    files = [f for f in files if os.path.basename(f) != "test_scens.json"]

    scenarios = []
    for f_path in files:
        with open(f_path, 'r') as f:
            data = json.load(f)
            scenarios.extend(data if isinstance(data, list) else [data])

    scenarios.sort(key=lambda x: x['scenario_id'])
    return scenarios


def process_scenarios(scen_range):
    g = load_graph()
    test_queries = load_test_queries(TEST_SCENS_PATH)
    scenarios = load_scenarios()

    for scen in scenarios:
        sid = scen['scenario_id']
        try:
            num = int(sid.split('-')[-1])
        except:
            continue

        if not (scen_range[0] <= num <= scen_range[1]):
            continue

        # Use queries from test_scens.json if available, otherwise fall back to scenario file
        if sid in test_queries:
            query_covered = test_queries[sid]['query_covered']
            query_denied = test_queries[sid]['query_denied']
            query_source = "test_scens.json"
        else:
            query_covered = scen['query_covered']
            query_denied = scen['query_denied']
            query_source = "scenario file"

        # Ground truth always comes from the scenario JSON files
        json_cov = [r['contract_id'] for r in scen['contract_responses'] if r['status'] == "COVERED"]
        json_den = [r['contract_id'] for r in scen['contract_responses'] if r['status'] == "DENIED"]
        json_na  = [r['contract_id'] for r in scen['contract_responses'] if r['status'] == "NOT_APPLICABLE"]

        res_cov = list(g.query(query_covered))
        res_den = list(g.query(query_denied))

        found_cov = {str(row[0].toPython()) for row in res_cov if row[0] is not None}
        found_den = {str(row[0].toPython()) for row in res_den if row[0] is not None}
        found_na  = ALL_CONTRACT_IDS - (found_cov | found_den)

        match_c  = (found_cov == set(json_cov))
        match_d  = (found_den == set(json_den))
        match_na = (found_na  == set(json_na))

        status = "PASS" if (match_c and match_d and match_na) else "FAIL"
        print(f"\n[{status}] {sid}: {scen['scenario_description'][:75]}...")
        print(f"        (queries from: {query_source})")
        print("-" * 80)

        if not match_c:
            print_pretty_sparql(query_covered)
        print_section("COVERED", json_cov, found_cov, res_cov)

        if not match_d:
            print_pretty_sparql(query_denied)
        print_section("DENIED", json_den, found_den, res_den)

        print_section("NOT_APPLICABLE (Calculated)", json_na, found_na)

        print("\n" + "." * 80)


In [6]:
process_scenarios((1, 58))

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<function _strToDecimal at 0x7279de17aac0>
Traceback (most recent call last):
  File "/home/mridum/miniconda3/envs/ontoscale/lib/python3.12/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
           ^^^^^^^^^^^^^^^^^^
  File "/home/mridum/miniconda3/envs/ontoscale/lib/python3.12/site-packages/owlrl/DatatypeHandling.py", line 132, in _strToDecimal
    return Decimal(v)
           ^^^^^^^^^^
decimal.InvalidOperation: [<class 'decimal.ConversionSyntax'>]
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<function _strToDecimal at 0x7279de17aac0>
Traceback (most recent call last):
  File "/home/mridum/miniconda3/envs/ontoscale/lib/python3.12/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignor

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function _strToDate at 0x7279de17b060>
Traceback (most recent call last):
  File "/home/mridum/miniconda3/envs/ontoscale/lib/python3.12/site-packages/owlrl/DatatypeHandling.py", line 442, in _strToDate
    tstr = time.strptime(final_v, "%Y-%m-%d")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mridum/miniconda3/envs/ontoscale/lib/python3.12/_strptime.py", line 647, in _strptime_time
    tt = _strptime(data_string, format)[0]
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mridum/miniconda3/envs/ontoscale/lib/python3.12/_strptime.py", line 432, in _strptime
    raise ValueError("time data %r does not match format %r" %
ValueError: time data 'the date the Master Policy terminates' does not match format '%Y-%m-%d'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/mridum/miniconda3/envs/ontoscale/

INITIALIZING: Loading Ontologies & Reasoning...
  (No TBox found at ../dataset/ontologies_eval/life_insurance_tbox.ttl, skipping)
  Loaded ABox: ../dataset/ontologies_eval/abox_C1.ttl
  Loaded ABox: ../dataset/ontologies_eval/abox_C10.ttl
  Loaded ABox: ../dataset/ontologies_eval/abox_C2.ttl
  Loaded ABox: ../dataset/ontologies_eval/abox_C3.ttl
  Loaded ABox: ../dataset/ontologies_eval/abox_C4.ttl
  Loaded ABox: ../dataset/ontologies_eval/abox_C5.ttl
  Loaded ABox: ../dataset/ontologies_eval/abox_C6.ttl
  Loaded ABox: ../dataset/ontologies_eval/abox_C7.ttl


Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<function _strToDecimal at 0x7279de17aac0>
Traceback (most recent call last):
  File "/home/mridum/miniconda3/envs/ontoscale/lib/python3.12/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
           ^^^^^^^^^^^^^^^^^^
  File "/home/mridum/miniconda3/envs/ontoscale/lib/python3.12/site-packages/owlrl/DatatypeHandling.py", line 132, in _strToDecimal
    return Decimal(v)
           ^^^^^^^^^^
decimal.InvalidOperation: [<class 'decimal.ConversionSyntax'>]
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<function _strToDecimal at 0x7279de17aac0>
Traceback (most recent call last):
  File "/home/mridum/miniconda3/envs/ontoscale/lib/python3.12/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignor

  Loaded ABox: ../dataset/ontologies_eval/abox_C8.ttl
  Loaded ABox: ../dataset/ontologies_eval/abox_C9.ttl
  Loaded ABox: ../dataset/ontologies_eval/life_insruance_tbox.ttl
  Running OWL-RL reasoning...
READY.


[PASS] SCEN-001: Insured dies from natural causes (heart attack) 5 years after policy issue....
        (queries from: test_scens.json)
--------------------------------------------------------------------------------

   ✅ COVERED:
      | C1 | N/A
      | C1 | misstatement of sex, death benefit adjustment, premium equivalence
      | C1 | Date of Birth, Insured Person, Policy Details
      | C1 | Insured Age, Underwriting Information, Policy Issuance Details
      | C1 | Insured Person, Policy Owner, Personal Information
      | C1 | Policy Owner, John Michael Anderson, Section 1
      | C1 | policy term, expiration date, coverage duration
      | C1 | premium amount, monthly payment, policy cost
      | C1 | premium payment mode, automatic deduction, monthly payments
      |

In [8]:
correct, total, pct, results = compute_scores(scenarios, g, test_queries, scen_range)
print_scores(correct, total, pct, results)

NameError: name 'scenarios' is not defined